### 028

In [153]:
# 965명의 유저가 있다.
# 01, 02 equipment 존재
# 01에는 F, L15, L30, R15, R30
# 02에는 F, L, R
print(965 * (5 + 3))
# F에선 2장씩, 나머지에선 1장씩
print(965 * (2 + 2 + 1 + 1 + 1 + 1 + 1 + 1))

7720
9650


In [10]:
import os
import pandas as pd
os.listdir('/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/028_data')
# meta data 가져오기

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment'

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/028_data/user_info_converted.csv"
label_df = pd.read_csv(label_path)
label_df['id'] = label_df['id'].apply(lambda x:str(x).zfill(4))
label_df = label_df[['id', 'l_cheek_pigmentation', 'r_cheek_pigmentation']]
label_df.rename(columns={'l_cheek_pigmentation': 'left_label', 'r_cheek_pigmentation': 'right_label'}, inplace=True)
label_df.head()

# 0-1은 0, 2-5는 1씩 빼기
label_df['left_label'] = label_df['left_label'].apply(lambda x: 0 if x <= 1 else x - 1)
label_df['right_label'] = label_df['right_label'].apply(lambda x: 0 if x <= 1 else x - 1)

label_df.head()

,id,left_label,right_label
0,0001,2,2
1,0002,2,2
2,0003,0,0
3,0004,2,2
4,0006,2,2


In [11]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment'

id_set = set()
exts = set()

for image in os.listdir(os.path.join(pig_root, '028_data/data')):
    exts.add(image.split('.')[-1])
    user_id = re.search(r'[0-9]{4}_[0-9]{2}_[A-Z][0-9]*.*', image).group(0)
    equipment = image.split('_')[2]
    id_set.add(user_id)
    
len(id_set), list(id_set)[:2], exts

(10615, ['0959_01_L15.png', '0756_02_R.jpg'], {'jpg', 'png'})

In [12]:
F01_df = pd.DataFrame(columns=['user_eq', 'left_path', 'right_path'])
F02_df = pd.DataFrame(columns=['user_eq', 'left_path', 'right_path'])
L15_df = pd.DataFrame(columns=['user_eq', 'left_path'])
L30_df = pd.DataFrame(columns=['user_eq', 'left_path'])
R15_df = pd.DataFrame(columns=['user_eq', 'right_path'])
R30_df = pd.DataFrame(columns=['user_eq', 'right_path'])
L_df = pd.DataFrame(columns=['user_eq', 'left_path'])
R_df = pd.DataFrame(columns=['user_eq', 'right_path'])

for user_id in id_set:
    if user_id.endswith('01_F.jpg'):
        F01_df.loc[len(F01_df)] = (user_id[:7], f"028_data/data/left_{user_id}", f"028_data/data/right_{user_id}")
    if user_id.endswith('02_F.jpg'):
        F02_df.loc[len(F02_df)] = (user_id[:7], f"028_data/data/left_{user_id}", f"028_data/data/right_{user_id}")
    if user_id.endswith('_L15.jpg'):
        L15_df.loc[len(L15_df)] = (user_id[:7], f"028_data/data/left_{user_id}")
    if user_id.endswith('_L30.jpg'):
        L30_df.loc[len(L30_df)] = (user_id[:7], f"028_data/data/left_{user_id}")
    if user_id.endswith('_R15.jpg'):
        R15_df.loc[len(R15_df)] = (user_id[:7], f"028_data/data/right_{user_id}")
    if user_id.endswith('_R30.jpg'):
        R30_df.loc[len(R30_df)] = (user_id[:7], f"028_data/data/right_{user_id}")
    if user_id.endswith('_L.jpg'):
        L_df.loc[len(L_df)] = (user_id[:7], f"028_data/data/left_{user_id}")
    if user_id.endswith('_R.jpg'):
        R_df.loc[len(R_df)] = (user_id[:7], f"028_data/data/right_{user_id}")
        
len(F01_df), len(F02_df), len(L15_df), len(L30_df), len(R15_df), len(R30_df), len(L_df), len(R_df)

(965, 965, 965, 965, 965, 965, 965, 965)

In [13]:
merged_15 = pd.merge(L15_df, R15_df, on='user_eq', how='outer')
merged_30 = pd.merge(L30_df, R30_df, on='user_eq', how='outer')
merged_LR = pd.merge(L_df, R_df, on='user_eq', how='outer')
len(merged_15), len(merged_30), len(merged_LR)

(965, 965, 965)

In [14]:
merged_df = pd.concat([F01_df, F02_df, merged_15, merged_30, merged_LR], ignore_index=True)
len(merged_df)

4825

In [15]:
merged_df['user_id'] = merged_df['user_eq'].apply(lambda x:x.split('_')[0])
merged_df['equipment'] = merged_df['user_eq'].apply(lambda x:x.split('_')[1])
merged_df.sample(3)

,user_eq,left_path,right_path,user_id,equipment
408,1076_01,028_data/data/left_1076_01_F.jpg,028_data/data/right_1076_01_F.jpg,1076,01
1097,0398_02,028_data/data/left_0398_02_F.jpg,028_data/data/right_0398_02_F.jpg,0398,02
2415,0562_01,028_data/data/left_0562_01_L15.jpg,028_data/data/right_0562_01_R15.jpg,0562,01


In [16]:
label_merged_df = pd.merge(merged_df, label_df, left_on='user_id', right_on='id')
label_merged_df.drop(columns=['id'], inplace=True)
label_merged_df = label_merged_df[['user_id', 'equipment', 'left_path', 'right_path', 'left_label', 'right_label']]
label_merged_df.sort_values(by=['user_id', 'equipment', 'left_path'], ignore_index=True, inplace=True)
label_merged_df.isnull().sum(), label_merged_df.isna().sum()

(user_id        0
 equipment      0
 left_path      0
 right_path     0
 left_label     0
 right_label    0
 dtype: int64,
 user_id        0
 equipment      0
 left_path      0
 right_path     0
 left_label     0
 right_label    0
 dtype: int64)

In [17]:
import json

json_path = os.path.join(pig_root, '028_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(label_merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 030

In [794]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/030_data/NIA_Korean_wholebody_pigmentation_250616.csv"
label_df = pd.read_csv(label_path)
label_df.rename(columns={'filename': 'user_id'})
label_df.drop(columns=['성별', '나이'], inplace=True)
label_df.head()

,filename,right_label,left_label
0,0001_face,0,0
1,0002_face,0,1
2,0003_face,1,0
3,0004_face,0,0
4,0005_face,1,1


In [795]:
len(os.listdir(os.path.join(pig_root, '030_data/data')))

3248

In [796]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '030_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'JPG', 'jpg'}

In [797]:
id_set = set()
image_df = pd.DataFrame(columns=['user_id', 'left_path', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '030_data/data')):
    image_path = os.path.join('030_data/data', image)
    user_id = image.split('_')[1]

    if user_id in image_df.loc[:, 'user_id'].values:
        if 'left' in image:
            image_df.loc[image_df['user_id'] == user_id, 'left_path'] = image_path
        elif 'right' in image:
            image_df.loc[image_df['user_id'] == user_id, 'right_path'] = image_path
        else:
            print(f"Unknown image format: {image}")
    else:
        if 'left' in image:
            image_df.loc[len(image_df)] = (user_id, image_path, None)
        elif 'right' in image:
            image_df.loc[len(image_df)] = (user_id, None, image_path)
        else:
            print(f"Unknown image format: {image}")

len(image_df)

1624

In [798]:
image_df.sample(3)

,user_id,left_path,right_path
472,0587,030_data/data/left_0587_face.jpg,030_data/data/right_0587_face.jpg
1113,1370,030_data/data/left_1370_face.jpg,030_data/data/right_1370_face.jpg
1274,1556,030_data/data/left_1556_face.jpg,030_data/data/right_1556_face.jpg


In [799]:
# image_df
label_df['user_id'] = label_df['filename'].apply(lambda x:x.split('_')[0])
label_df

,filename,right_label,left_label,user_id
0,0001_face,0,0,0001
1,0002_face,0,1,0002
2,0003_face,1,0,0003
3,0004_face,0,0,0004
4,0005_face,1,1,0005
...,...,...,...,...
1619,1976_face,2,1,1976
1620,1977_face,2,3,1977
1621,1978_face,1,2,1978
1622,1979_face,4,4,1979


In [800]:
merged_df = pd.merge(image_df, label_df, on='user_id')
len(merged_df)

1624

In [801]:
failed_paths = set()
for idx, row in merged_df.iterrows():
    if not os.path.exists(os.path.join(pig_root, row['left_path'])):
        failed_paths.add(row['left_path'])
    if not os.path.exists(os.path.join(pig_root, row['right_path'])):
        failed_paths.add(row['right_path'])

len(failed_paths)

0

In [806]:
merged_df = merged_df[['user_id', 'left_path', 'right_path', 'left_label', 'right_label']]
merged_df

,user_id,left_path,right_path,left_label,right_label
0,0001,030_data/data/left_0001_face.jpg,030_data/data/right_0001_face.jpg,0,0
1,0002,030_data/data/left_0002_face.jpg,030_data/data/right_0002_face.jpg,1,0
2,0005,030_data/data/left_0005_face.jpg,030_data/data/right_0005_face.jpg,1,1
3,0003,030_data/data/left_0003_face.jpg,030_data/data/right_0003_face.jpg,0,1
4,0007,030_data/data/left_0007_face.JPG,030_data/data/right_0007_face.JPG,0,0
...,...,...,...,...,...
1619,1977,030_data/data/left_1977_face.jpg,030_data/data/right_1977_face.jpg,3,2
1620,1976,030_data/data/left_1976_face.jpg,030_data/data/right_1976_face.jpg,1,2
1621,1978,030_data/data/left_1978_face.jpg,030_data/data/right_1978_face.jpg,2,1
1622,1979,030_data/data/left_1979_face.jpg,030_data/data/right_1979_face.jpg,4,4


In [807]:
merged_df.drop(columns=['filename'], inplace=True)

KeyError: "['filename'] not found in axis"

In [808]:
import json

json_path = os.path.join(pig_root, '030_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 034

In [188]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/034_data/results_250507.csv"
label_df = pd.read_csv(label_path)

label_df.head()

,filename,label
0,left_cheek_0000_CRS_19_01.jpg,0
1,left_cheek_0000_CRS_46_01.jpg,0
2,left_cheek_0001_CRS_19_01.jpg,1
3,left_cheek_0001_CRS_46_01.jpg,1
4,left_cheek_0002_CRS_19_01.jpg,0


In [210]:
label_df['user_id'] = label_df['filename'].apply(lambda x: x[x.find('cheek_')+6:-4])
left_label_df = label_df[label_df['filename'].str.contains('left_cheek')].sort_values(by='user_id', ignore_index=True)
right_label_df = label_df[label_df['filename'].str.contains('right_cheek')].sort_values(by='user_id', ignore_index=True)

In [216]:
left_label_df.rename(columns={'label': 'left_label'}, inplace=True)
right_label_df.rename(columns={'label': 'right_label'}, inplace=True)

label_df = pd.merge(left_label_df[['user_id', 'filename', 'left_label']], right_label_df[['user_id', 'right_label']], on='user_id')
label_df

,user_id,filename,left_label,right_label
0,0000_CRS_19_01,left_cheek_0000_CRS_19_01.jpg,0,0
1,0000_CRS_46_01,left_cheek_0000_CRS_46_01.jpg,0,0
2,0001_CRS_19_01,left_cheek_0001_CRS_19_01.jpg,1,0
3,0001_CRS_46_01,left_cheek_0001_CRS_46_01.jpg,1,0
4,0002_CRS_19_01,left_cheek_0002_CRS_19_01.jpg,0,1
...,...,...,...,...
5395,2697_CRS_46_01,left_cheek_2697_CRS_46_01.jpg,0,0
5396,2698_CRS_19_01,left_cheek_2698_CRS_19_01.jpg,1,1
5397,2698_CRS_46_01,left_cheek_2698_CRS_46_01.jpg,1,1
5398,2699_CRS_19_01,left_cheek_2699_CRS_19_01.jpg,2,2


In [217]:
len(os.listdir(os.path.join(pig_root, '034_data/data')))

10800

In [218]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '034_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'JPG', 'jpg'}

In [239]:
left_image_df = pd.DataFrame(columns=['user_id', 'left_path'])
right_image_df = pd.DataFrame(columns=['user_id', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '034_data/data')):
    image_path = os.path.join('034_data/data', image)
    if 'left' in image:
        user_id = image[5:-4]
        left_image_df.loc[len(left_image_df)] = (user_id, image_path)
    elif 'right' in image:
        user_id = image[6:-4]
        right_image_df.loc[len(right_image_df)] = (user_id, image_path)
    else:
        print(f"Unknown image format: {image}")
        continue

len(left_image_df), len(right_image_df)

(5400, 5400)

In [244]:
image_df = pd.merge(left_image_df, right_image_df, on='user_id', how='outer')

In [245]:
image_df.isna().sum(), image_df.isnull().sum()

(user_id       0
 left_path     0
 right_path    0
 dtype: int64,
 user_id       0
 left_path     0
 right_path    0
 dtype: int64)

In [252]:
# image_df
label_df.drop(columns=['filename'], inplace=True)
label_df

,user_id,left_label,right_label
0,0000_CRS_19_01,0,0
1,0000_CRS_46_01,0,0
2,0001_CRS_19_01,1,0
3,0001_CRS_46_01,1,0
4,0002_CRS_19_01,0,1
...,...,...,...
5395,2697_CRS_46_01,0,0
5396,2698_CRS_19_01,1,1
5397,2698_CRS_46_01,1,1
5398,2699_CRS_19_01,2,2


In [253]:
merged_df = pd.merge(image_df, label_df, on='user_id')
len(merged_df)

5400

In [254]:
merged_df

,user_id,left_path,right_path,left_label,right_label
0,0000_CRS_19_01,034_data/data/left_0000_CRS_19_01.jpg,034_data/data/right_0000_CRS_19_01.jpg,0,0
1,0000_CRS_46_01,034_data/data/left_0000_CRS_46_01.jpg,034_data/data/right_0000_CRS_46_01.jpg,0,0
2,0001_CRS_19_01,034_data/data/left_0001_CRS_19_01.jpg,034_data/data/right_0001_CRS_19_01.jpg,1,0
3,0001_CRS_46_01,034_data/data/left_0001_CRS_46_01.jpg,034_data/data/right_0001_CRS_46_01.jpg,1,0
4,0002_CRS_19_01,034_data/data/left_0002_CRS_19_01.jpg,034_data/data/right_0002_CRS_19_01.jpg,0,1
...,...,...,...,...,...
5395,2697_CRS_46_01,034_data/data/left_2697_CRS_46_01.jpg,034_data/data/right_2697_CRS_46_01.jpg,0,0
5396,2698_CRS_19_01,034_data/data/left_2698_CRS_19_01.JPG,034_data/data/right_2698_CRS_19_01.JPG,1,1
5397,2698_CRS_46_01,034_data/data/left_2698_CRS_46_01.JPG,034_data/data/right_2698_CRS_46_01.JPG,1,1
5398,2699_CRS_19_01,034_data/data/left_2699_CRS_19_01.jpg,034_data/data/right_2699_CRS_19_01.jpg,2,2


In [258]:
merged_df['real_id'] = merged_df['user_id']

In [260]:
merged_df['user_id'] = merged_df['user_id'].apply(lambda x: x.split('_')[0])
merged_df

,user_id,left_path,right_path,left_label,right_label,real_id
0,0000,034_data/data/left_0000_CRS_19_01.jpg,034_data/data/right_0000_CRS_19_01.jpg,0,0,0000_CRS_19_01
1,0000,034_data/data/left_0000_CRS_46_01.jpg,034_data/data/right_0000_CRS_46_01.jpg,0,0,0000_CRS_46_01
2,0001,034_data/data/left_0001_CRS_19_01.jpg,034_data/data/right_0001_CRS_19_01.jpg,1,0,0001_CRS_19_01
3,0001,034_data/data/left_0001_CRS_46_01.jpg,034_data/data/right_0001_CRS_46_01.jpg,1,0,0001_CRS_46_01
4,0002,034_data/data/left_0002_CRS_19_01.jpg,034_data/data/right_0002_CRS_19_01.jpg,0,1,0002_CRS_19_01
...,...,...,...,...,...,...
5395,2697,034_data/data/left_2697_CRS_46_01.jpg,034_data/data/right_2697_CRS_46_01.jpg,0,0,2697_CRS_46_01
5396,2698,034_data/data/left_2698_CRS_19_01.JPG,034_data/data/right_2698_CRS_19_01.JPG,1,1,2698_CRS_19_01
5397,2698,034_data/data/left_2698_CRS_46_01.JPG,034_data/data/right_2698_CRS_46_01.JPG,1,1,2698_CRS_46_01
5398,2699,034_data/data/left_2699_CRS_19_01.jpg,034_data/data/right_2699_CRS_19_01.jpg,2,2,2699_CRS_19_01


In [261]:
merged_df = merged_df[['user_id', 'real_id', 'left_path', 'right_path', 'left_label', 'right_label']]

In [262]:
failed_paths = set()
for idx, row in merged_df.iterrows():
    if not os.path.exists(os.path.join(pig_root, row['left_path'])):
        failed_paths.add(row['left_path'])
    if not os.path.exists(os.path.join(pig_root, row['right_path'])):
        failed_paths.add(row['right_path'])

len(failed_paths)

0

In [263]:
import json

json_path = os.path.join(pig_root, '034_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 045

In [644]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/045_data/NIA_family_relation_pigmentation_250612.csv"
label_df = pd.read_csv(label_path)

label_df.head()

,filename,right_label,left_label
0,F0001_IND_D_18_-45_01.JPG,NaN,0.0
1,F0001_IND_D_18_-45_02.JPG,NaN,0.0
2,F0001_IND_D_18_0_01.JPG,0.0,0.0
3,F0001_IND_D_18_0_02.JPG,0.0,0.0
4,F0001_IND_D_18_45_01.JPG,0.0,NaN


In [645]:
len(label_df)

4622

In [646]:
!find /home/work/hocheol_dir/workspace/datasets/v0.2_data/_origin_data/045_data/data -maxdepth 1 -type f | wc -l

4622


In [647]:
import re

left_label_df = pd.DataFrame(columns=['filename', 'left_label'])
right_label_df = pd.DataFrame(columns=['filename', 'right_label'])
F_label_df = pd.DataFrame(columns=['filename', 'left_label', 'right_label'])

for i, row in label_df.iterrows():
    # print(row['filename'])
    # img_id = re.search(r'F[0-9]{4}_[A-Z]*_[A-Z]*', row['filename']).group(0)
    if '_-45_' in row['filename']:
        # left
        left_label_df.loc[len(left_label_df)] = (row['filename'], row['left_label'])
    elif '_45_' in row['filename']:
        # right
        right_label_df.loc[len(right_label_df)] = (row['filename'], row['right_label'])
    elif '_0_' in row['filename']:
        # 정면
        F_label_df.loc[len(F_label_df)] = (row['filename'], row['left_label'], row['right_label'])
    else:
        print(f"Unknown image format: {row['filename']}")

len(left_label_df), len(right_label_df), len(F_label_df), len(label_df)

(1407, 1456, 1759, 4622)

In [648]:
sum([1407, 1456, 1759])

4622

In [649]:
left_label_df

,filename,left_label
0,F0001_IND_D_18_-45_01.JPG,0.0
1,F0001_IND_D_18_-45_02.JPG,0.0
2,F0001_IND_GM_75_-45_01.JPG,4.0
3,F0001_IND_GM_75_-45_02.JPG,4.0
4,F0001_IND_M_45_-45_01.JPG,1.0
...,...,...
1402,F0897_IND_M_50_-45_02.JPG,2.0
1403,F0898_IND_D2_14_-45_02.JPG,0.0
1404,F0898_IND_D_19_-45_02.JPG,0.0
1405,F0898_IND_F_50_-45_02.JPG,1.0


In [650]:
len(os.listdir(os.path.join(pig_root, '045_data/data')))

6437

In [651]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '045_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'JPG', 'jpg'}

In [652]:
F_image_df = pd.DataFrame(columns=['filename', 'left_path', 'right_path'])
L_image_df = pd.DataFrame(columns=['filename', 'left_path'])
R_image_df = pd.DataFrame(columns=['filename', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '045_data/data')):
    image_path = os.path.join('045_data/data', image)
    filename = re.search(r'F[0-9]{4}_[A-Z]+.*', image).group(0)
    if '_-45_' in image:
        L_image_df.loc[len(L_image_df)] = (filename, image_path)
    elif '_45_' in image:
        R_image_df.loc[len(R_image_df)] = (filename, image_path)
    elif '_0_' in image:
        if filename not in F_image_df['filename'].values:
            if 'left' in image:
                F_image_df.loc[len(F_image_df)] = (filename, image_path, None)
            else:
                F_image_df.loc[len(F_image_df)] = (filename, None, image_path)
        else:
            if 'left' in image:
                F_image_df.loc[F_image_df['filename'] == filename, 'left_path'] = image_path
            else:
                F_image_df.loc[F_image_df['filename'] == filename, 'right_path'] = image_path

    else:
        print(f"Unknown image format: {image}")

len(F_image_df), len(L_image_df), len(R_image_df)

(1759, 1405, 1514)

In [653]:
F_image_df

,filename,left_path,right_path
0,F0001_IND_D_18_0_02.JPG,045_data/data/left_F0001_IND_D_18_0_02.JPG,045_data/data/right_F0001_IND_D_18_0_02.JPG
1,F0001_IND_D_18_0_01.JPG,045_data/data/left_F0001_IND_D_18_0_01.JPG,045_data/data/right_F0001_IND_D_18_0_01.JPG
2,F0001_IND_GM_75_0_01.JPG,045_data/data/left_F0001_IND_GM_75_0_01.JPG,045_data/data/right_F0001_IND_GM_75_0_01.JPG
3,F0002_IND_S_16_0_01.JPG,045_data/data/left_F0002_IND_S_16_0_01.JPG,045_data/data/right_F0002_IND_S_16_0_01.JPG
4,F0001_IND_GM_75_0_02.JPG,045_data/data/left_F0001_IND_GM_75_0_02.JPG,045_data/data/right_F0001_IND_GM_75_0_02.JPG
...,...,...,...
1754,F0897_IND_M_50_0_02.JPG,045_data/data/left_F0897_IND_M_50_0_02.JPG,045_data/data/right_F0897_IND_M_50_0_02.JPG
1755,F0898_IND_D2_14_0_02.JPG,045_data/data/left_F0898_IND_D2_14_0_02.JPG,045_data/data/right_F0898_IND_D2_14_0_02.JPG
1756,F0898_IND_F_50_0_02.JPG,045_data/data/left_F0898_IND_F_50_0_02.JPG,045_data/data/right_F0898_IND_F_50_0_02.JPG
1757,F0898_IND_D_19_0_02.JPG,045_data/data/left_F0898_IND_D_19_0_02.JPG,045_data/data/right_F0898_IND_D_19_0_02.JPG


In [654]:
left_merged = pd.merge(left_label_df, L_image_df, on='filename', how='outer')
left_merged.isna().sum()

filename      0
left_label    0
left_path     2
dtype: int64

In [655]:
right_merged = pd.merge(right_label_df, R_image_df, on='filename', how='outer')
right_merged.isna().sum()

filename       0
right_label    0
right_path     2
dtype: int64

In [656]:
F_merged = pd.merge(F_label_df, F_image_df, on='filename', how='outer')
F_merged.isna().sum()

filename       0
left_label     0
right_label    0
left_path      0
right_path     0
dtype: int64

In [657]:
F_merged = F_merged[['filename', 'left_path', 'right_path', 'left_label', 'right_label']]
F_merged

,filename,left_path,right_path,left_label,right_label
0,F0001_IND_D_18_0_01.JPG,045_data/data/left_F0001_IND_D_18_0_01.JPG,045_data/data/right_F0001_IND_D_18_0_01.JPG,0.0,0.0
1,F0001_IND_D_18_0_02.JPG,045_data/data/left_F0001_IND_D_18_0_02.JPG,045_data/data/right_F0001_IND_D_18_0_02.JPG,0.0,0.0
2,F0001_IND_GM_75_0_01.JPG,045_data/data/left_F0001_IND_GM_75_0_01.JPG,045_data/data/right_F0001_IND_GM_75_0_01.JPG,4.0,4.0
3,F0001_IND_GM_75_0_02.JPG,045_data/data/left_F0001_IND_GM_75_0_02.JPG,045_data/data/right_F0001_IND_GM_75_0_02.JPG,4.0,4.0
4,F0002_IND_S_16_0_01.JPG,045_data/data/left_F0002_IND_S_16_0_01.JPG,045_data/data/right_F0002_IND_S_16_0_01.JPG,0.0,0.0
...,...,...,...,...,...
1754,F0897_IND_M_50_0_02.JPG,045_data/data/left_F0897_IND_M_50_0_02.JPG,045_data/data/right_F0897_IND_M_50_0_02.JPG,2.0,2.0
1755,F0898_IND_D2_14_0_02.JPG,045_data/data/left_F0898_IND_D2_14_0_02.JPG,045_data/data/right_F0898_IND_D2_14_0_02.JPG,0.0,0.0
1756,F0898_IND_D_19_0_02.JPG,045_data/data/left_F0898_IND_D_19_0_02.JPG,045_data/data/right_F0898_IND_D_19_0_02.JPG,0.0,0.0
1757,F0898_IND_F_50_0_02.JPG,045_data/data/left_F0898_IND_F_50_0_02.JPG,045_data/data/right_F0898_IND_F_50_0_02.JPG,1.0,2.0


In [658]:
# LR_merged 만들자
left_merged['img_id'] = left_merged['filename'].apply(lambda x: x.replace('_-45_', '_LR_'))
right_merged['img_id'] = right_merged['filename'].apply(lambda x: x.replace('_45_', '_LR_'))
LR_merged = pd.merge(left_merged, right_merged, on='img_id', how='outer')
# LR_merged.rename(columns={'filename_x': 'left_filename', 'filename_y': 'right_filename'}, inplace=True)
LR_merged = LR_merged[['img_id', 'left_path', 'right_path', 'left_label', 'right_label']].rename(columns={'img_id': 'filename'})
LR_merged

,filename,left_path,right_path,left_label,right_label
0,F0001_IND_D_18_LR_01.JPG,045_data/data/left_F0001_IND_D_18_-45_01.JPG,045_data/data/right_F0001_IND_D_18_45_01.JPG,0.0,0.0
1,F0001_IND_D_18_LR_02.JPG,045_data/data/left_F0001_IND_D_18_-45_02.JPG,045_data/data/right_F0001_IND_D_18_45_02.JPG,0.0,0.0
2,F0001_IND_GM_75_LR_01.JPG,045_data/data/left_F0001_IND_GM_75_-45_01.JPG,045_data/data/right_F0001_IND_GM_75_45_01.JPG,4.0,4.0
3,F0001_IND_GM_75_LR_02.JPG,045_data/data/left_F0001_IND_GM_75_-45_02.JPG,045_data/data/right_F0001_IND_GM_75_45_02.JPG,4.0,4.0
4,F0001_IND_M_45_LR_01.JPG,045_data/data/left_F0001_IND_M_45_-45_01.JPG,NaN,1.0,NaN
...,...,...,...,...,...
1592,F0897_IND_M_50_LR_02.JPG,045_data/data/left_F0897_IND_M_50_-45_02.JPG,045_data/data/right_F0897_IND_M_50_45_02.JPG,2.0,2.0
1593,F0898_IND_D2_14_LR_02.JPG,045_data/data/left_F0898_IND_D2_14_-45_02.JPG,045_data/data/right_F0898_IND_D2_14_45_02.JPG,0.0,0.0
1594,F0898_IND_D_19_LR_02.JPG,045_data/data/left_F0898_IND_D_19_-45_02.JPG,045_data/data/right_F0898_IND_D_19_45_02.JPG,0.0,0.0
1595,F0898_IND_F_50_LR_02.JPG,045_data/data/left_F0898_IND_F_50_-45_02.JPG,045_data/data/right_F0898_IND_F_50_45_02.JPG,1.0,2.0


In [659]:
merged_df = pd.concat([F_merged, LR_merged], ignore_index=True)


In [661]:
import re

merged_df['user_id'] = merged_df['filename'].apply(lambda x:re.search(r'F[0-9]{4}_[A-Z]+_[A-Z]+[0-9]*', x).group(0))
merged_df

,filename,left_path,right_path,left_label,right_label,user_id
0,F0001_IND_D_18_0_01.JPG,045_data/data/left_F0001_IND_D_18_0_01.JPG,045_data/data/right_F0001_IND_D_18_0_01.JPG,0.0,0.0,F0001_IND_D
1,F0001_IND_D_18_0_02.JPG,045_data/data/left_F0001_IND_D_18_0_02.JPG,045_data/data/right_F0001_IND_D_18_0_02.JPG,0.0,0.0,F0001_IND_D
2,F0001_IND_GM_75_0_01.JPG,045_data/data/left_F0001_IND_GM_75_0_01.JPG,045_data/data/right_F0001_IND_GM_75_0_01.JPG,4.0,4.0,F0001_IND_GM
3,F0001_IND_GM_75_0_02.JPG,045_data/data/left_F0001_IND_GM_75_0_02.JPG,045_data/data/right_F0001_IND_GM_75_0_02.JPG,4.0,4.0,F0001_IND_GM
4,F0002_IND_S_16_0_01.JPG,045_data/data/left_F0002_IND_S_16_0_01.JPG,045_data/data/right_F0002_IND_S_16_0_01.JPG,0.0,0.0,F0002_IND_S
...,...,...,...,...,...,...
3351,F0897_IND_M_50_LR_02.JPG,045_data/data/left_F0897_IND_M_50_-45_02.JPG,045_data/data/right_F0897_IND_M_50_45_02.JPG,2.0,2.0,F0897_IND_M
3352,F0898_IND_D2_14_LR_02.JPG,045_data/data/left_F0898_IND_D2_14_-45_02.JPG,045_data/data/right_F0898_IND_D2_14_45_02.JPG,0.0,0.0,F0898_IND_D2
3353,F0898_IND_D_19_LR_02.JPG,045_data/data/left_F0898_IND_D_19_-45_02.JPG,045_data/data/right_F0898_IND_D_19_45_02.JPG,0.0,0.0,F0898_IND_D
3354,F0898_IND_F_50_LR_02.JPG,045_data/data/left_F0898_IND_F_50_-45_02.JPG,045_data/data/right_F0898_IND_F_50_45_02.JPG,1.0,2.0,F0898_IND_F


In [664]:
merged_df = merged_df[['user_id', 'filename', 'left_path', 'right_path', 'left_label', 'right_label']]
merged_df.head(3)

,user_id,filename,left_path,right_path,left_label,right_label
0,F0001_IND_D,F0001_IND_D_18_0_01.JPG,045_data/data/left_F0001_IND_D_18_0_01.JPG,045_data/data/right_F0001_IND_D_18_0_01.JPG,0.0,0.0
1,F0001_IND_D,F0001_IND_D_18_0_02.JPG,045_data/data/left_F0001_IND_D_18_0_02.JPG,045_data/data/right_F0001_IND_D_18_0_02.JPG,0.0,0.0
2,F0001_IND_GM,F0001_IND_GM_75_0_01.JPG,045_data/data/left_F0001_IND_GM_75_0_01.JPG,045_data/data/right_F0001_IND_GM_75_0_01.JPG,4.0,4.0


In [665]:
failed_paths = set()

for i, row in merged_df.iterrows():
    if str(row['left_path']) != 'nan':
        if not os.path.exists(os.path.join(pig_root, row['left_path'])):
            print(f"Left path does not exist: {row['left_path']}")
            failed_paths.add(row['left_path'])
    else:
        failed_paths.add(row['left_path'])
    if str(row['right_path']) != 'nan':
        if not os.path.exists(os.path.join(pig_root, row['right_path'])):
            print(f"Right path does not exist: {row['right_path']}")
            failed_paths.add(row['right_path'])
    else:
        failed_paths.add(row['right_path'])
len(failed_paths)

1

In [666]:
import json

json_path = os.path.join(pig_root, '045_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 118_data (고화질 only)

In [809]:
import os
import pandas as pd
# meta data 가져오기

label_path = "/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/118_data/NIA_age_face_pigmentation_250619.csv"
label_df = pd.read_csv(label_path)
label_df.head()

,filename,right_cheek,left_cheek,box_w,box_h
0,0001_1992_28_00000071_D.png,NaN,NaN,607.442444,721.450480
1,0002_1997_26_00000050_D.png,0.0,0.0,857.429579,957.556410
2,0002_1997_26_00000051_D.png,0.0,0.0,837.935589,900.689212
3,0003_1988_20_00000038_D.png,NaN,NaN,657.667152,728.690622
4,0003_1988_20_00000046_D.png,NaN,NaN,816.169738,902.846387


In [810]:
import math
label_df['valid'] = label_df.apply(lambda x: 0 if math.isnan(x['right_cheek']) or math.isnan(x['left_cheek']) else 1, axis=1)
label_df = label_df[label_df['valid'] == 1]

In [811]:
len(os.listdir(os.path.join(pig_root, '118_data/data')))

4964

In [812]:
label_df = label_df[['filename', 'right_cheek', 'left_cheek']]
label_df[['right_cheek', 'left_cheek']] = label_df[['right_cheek', 'left_cheek']].astype(int)
label_df

,filename,right_cheek,left_cheek
1,0002_1997_26_00000050_D.png,0,0
2,0002_1997_26_00000051_D.png,0,0
7,0003_1988_35_00000059_D.png,0,0
21,0010_1997_11_00000039_D.png,0,0
39,0013_1972_46_00000035_D.png,0,0
...,...,...,...
2455,0995_2003_20_00000039_D.png,0,0
2456,0995_2003_20_00000040_D.png,0,0
2457,0995_2003_20_00000042_D.png,0,0
2458,0995_2003_20_00000044_D.png,0,0


In [813]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '118_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'png'}

In [814]:
id_set = set()
image_df = pd.DataFrame(columns=['filename', 'left_path', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '118_data/data')):
    image_path = os.path.join('118_data/data', image)
    filename = image.replace('left_', '').replace('right_', '')
    if filename in label_df['filename'].values:
        if 'left' in image:
            if filename not in image_df['filename'].values:
                image_df.loc[len(image_df)] = (filename, image_path, None)
            else:
                image_df.loc[image_df['filename'] == filename, 'left_path'] = image_path
        elif 'right' in image:
            if filename not in image_df['filename'].values:
                image_df.loc[len(image_df)] = (filename, None, image_path)
            else:
                image_df.loc[image_df['filename'] == filename, 'right_path'] = image_path

len(image_df)

775

In [815]:
# image_df
label_df

,filename,right_cheek,left_cheek
1,0002_1997_26_00000050_D.png,0,0
2,0002_1997_26_00000051_D.png,0,0
7,0003_1988_35_00000059_D.png,0,0
21,0010_1997_11_00000039_D.png,0,0
39,0013_1972_46_00000035_D.png,0,0
...,...,...,...
2455,0995_2003_20_00000039_D.png,0,0
2456,0995_2003_20_00000040_D.png,0,0
2457,0995_2003_20_00000042_D.png,0,0
2458,0995_2003_20_00000044_D.png,0,0


In [816]:
merged_df = pd.merge(image_df, label_df, on='filename', how='inner')
len(merged_df)

775

In [817]:
failed_paths = set()
for idx, row in merged_df.iterrows():
    if not os.path.exists(os.path.join(pig_root, row['left_path'])):
        failed_paths.add(row['left_path'])
    if not os.path.exists(os.path.join(pig_root, row['right_path'])):
        failed_paths.add(row['right_path'])

len(failed_paths)

0

In [818]:
# 같은사람 연도 바뀌어도(다른 나이) 같은 user_id로 묶기
merged_df['user_id'] = merged_df['filename'].apply(lambda x:x.split('_')[0])
merged_df = merged_df[['user_id', 'filename', 'left_path', 'right_path', 'left_cheek', 'right_cheek']]
merged_df.sort_values(by=['user_id', 'filename'], ignore_index=True, inplace=True)

In [819]:
merged_df.rename(columns={'left_cheek': 'left_label', 'right_cheek': 'right_label'}, inplace=True)

In [820]:
import json

json_path = os.path.join(pig_root, '118_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)

### 999

In [705]:
import os
import pandas as pd
# meta data 가져오기

label_paths = [
    '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/999_data/NIA_online_pigment.csv',
    '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment/999_data/NIA_SNUH_pigment.csv'
]
label_df = pd.concat([pd.read_csv(path) for path in label_paths], ignore_index=True)
label_df.head()

,filename,right_cheek,left_cheek
0,6101_4_F,0.0,0.0
1,6101_4_F_2,0.0,0.0
2,6101_4_L,NaN,0.0
3,6101_4_L_2,NaN,0.0
4,6101_4_R,0.0,NaN


In [706]:
len(label_df)

5826

In [707]:
len(os.listdir(os.path.join(pig_root, '999_data/data')))

7789

In [708]:
# image path 가져오기
import os
import re

pig_root = '/home/work/hocheol_dir/workspace/datasets/v0.2_data/pigment'

exts = set()

for image in os.listdir(os.path.join(pig_root, '999_data/data')):
    exts.add(image.split('.')[-1])
    
exts

{'JPG', 'jpeg', 'jpg', 'png'}

In [711]:
label_df.sample(3)

,filename,right_cheek,left_cheek
3282,6241_4_F_2,0.0,1.0
2383,6202_5_L,NaN,2.0
2543,6209_4_F,2.0,2.0


In [720]:
len(label_df)

5826

In [725]:
import os

F_img_df = pd.DataFrame(columns=['filename', 'left_path', 'right_path'])
L_img_df = pd.DataFrame(columns=['filename', 'left_path'])
R_img_df = pd.DataFrame(columns=['filename', 'right_path'])

for image in os.listdir(os.path.join(pig_root, '999_data/data')):
    image_path = os.path.join('999_data/data', image)
    filename = os.path.splitext(image.replace('left_', '').replace('right_', ''))[0]
    if '_F' in filename:
        if 'left' in image:
            if filename not in F_img_df['filename'].values:
                F_img_df.loc[len(F_img_df)] = (filename, image_path, None)
            else:
                F_img_df.loc[F_img_df['filename'] == filename, 'left_path'] = image_path
        elif 'right' in image:
            if filename not in F_img_df['filename'].values:
                F_img_df.loc[len(F_img_df)] = (filename, None, image_path)
            else:
                F_img_df.loc[F_img_df['filename'] == filename, 'right_path'] = image_path
    elif '_L' in filename:
        if 'left' in image:
            L_img_df.loc[len(L_img_df)] = (filename, image_path)
        else:
            print(f"Unknown image format: {image}")
    elif '_R' in filename:
        if 'right' in image:
            R_img_df.loc[len(R_img_df)] = (filename, image_path)
        else:
            print(f"Unknown image format: {image}")
    else:
        print(f"Unknown image format: {image}")

len(F_img_df), len(L_img_df), len(R_img_df)

(1996, 1879, 1919)

In [778]:
F_merged = pd.merge(F_img_df, label_df, on='filename', how='inner')
F_merged['user_id'] = F_merged['filename'].apply(lambda x: x.split('_')[0])
F_merged.rename(columns={'left_cheek': 'left_label', 'right_cheek': 'right_label'}, inplace=True)
F_merged = F_merged[['user_id', 'filename', 'left_path', 'right_path', 'left_label', 'right_label']]
F_merged

,user_id,filename,left_path,right_path,left_label,right_label
0,6101,6101_5_F,999_data/data/left_6101_5_F.jpg,999_data/data/right_6101_5_F.jpg,0.0,0.0
1,6101,6101_4_F_2,999_data/data/left_6101_4_F_2.jpg,999_data/data/right_6101_4_F_2.jpg,0.0,0.0
2,6101,6101_4_F,999_data/data/left_6101_4_F.jpg,999_data/data/right_6101_4_F.jpg,0.0,0.0
3,6101,6101_5_F_2,999_data/data/left_6101_5_F_2.jpg,999_data/data/right_6101_5_F_2.jpg,0.0,0.0
4,6101,6101_7_F,999_data/data/left_6101_7_F.jpg,999_data/data/right_6101_7_F.jpg,0.0,0.0
...,...,...,...,...,...,...
1575,6302,6302_7_F_2,999_data/data/left_6302_7_F_2.jpg,999_data/data/right_6302_7_F_2.jpg,0.0,0.0
1576,6302,6302_7_F,999_data/data/left_6302_7_F.jpg,999_data/data/right_6302_7_F.jpg,0.0,0.0
1577,6302,6302_4_F,999_data/data/left_6302_4_F.jpg,999_data/data/right_6302_4_F.jpg,0.0,0.0
1578,6302,6302_6_F_2,999_data/data/left_6302_6_F_2.jpg,999_data/data/right_6302_6_F_2.jpg,0.0,0.0


In [779]:
L_merged = pd.merge(L_img_df, label_df, on='filename', how='inner').drop(columns=['right_cheek'])

In [780]:
R_merged = pd.merge(R_img_df, label_df, on='filename', how='inner').drop(columns=['left_cheek'])

In [781]:
R_merged.head(2)

,filename,right_path,id,right_cheek
0,6101_5_R,999_data/data/right_6101_5_R.jpg,6101_5_LR,0.0
1,6101_4_R_2,999_data/data/right_6101_4_R_2.jpg,6101_4_LR_2,0.0


In [782]:
L_merged['id'] = L_merged['filename'].apply(lambda x:x.replace('_L', '_LR'))
R_merged['id'] = R_merged['filename'].apply(lambda x:x.replace('_R', '_LR'))
LR_merged = pd.merge(L_merged, R_merged, on='id', how='outer')
LR_merged.drop(columns=['filename_x', 'filename_y'], inplace=True)
LR_merged.rename(columns={'id':'filename'}, inplace=True)
LR_merged['user_id'] = LR_merged['filename'].apply(lambda x:x.split('_')[0])
LR_merged = LR_merged[['user_id', 'filename', 'left_path', 'right_path','left_cheek', 'right_cheek']]
LR_merged.rename(columns={'left_cheek': 'left_label', 'right_cheek': 'right_label'}, inplace=True)
LR_merged

,user_id,filename,left_path,right_path,left_label,right_label
0,6101,6101_4_LR,999_data/data/left_6101_4_L.jpg,999_data/data/right_6101_4_R.jpg,0.0,0.0
1,6101,6101_4_LR_2,999_data/data/left_6101_4_L_2.jpg,999_data/data/right_6101_4_R_2.jpg,0.0,0.0
2,6101,6101_5_LR,999_data/data/left_6101_5_L.jpg,999_data/data/right_6101_5_R.jpg,0.0,0.0
3,6101,6101_5_LR_2,999_data/data/left_6101_5_L_2.jpg,999_data/data/right_6101_5_R_2.jpg,0.0,0.0
4,6101,6101_6_LR,999_data/data/left_6101_6_L.jpg,999_data/data/right_6101_6_R.jpg,0.0,0.0
...,...,...,...,...,...,...
1575,6302,6302_5_LR_2,999_data/data/left_6302_5_L_2.jpg,999_data/data/right_6302_5_R_2.jpg,0.0,0.0
1576,6302,6302_6_LR,999_data/data/left_6302_6_L.jpg,999_data/data/right_6302_6_R.jpg,0.0,0.0
1577,6302,6302_6_LR_2,999_data/data/left_6302_6_L_2.jpg,999_data/data/right_6302_6_R_2.jpg,0.0,0.0
1578,6302,6302_7_LR,999_data/data/left_6302_7_L.jpg,999_data/data/right_6302_7_R.jpg,0.0,0.0


In [783]:
LR_merged.sample(3)

,user_id,filename,left_path,right_path,left_label,right_label
249,6133,6133_4_LR_2,999_data/data/left_6133_4_L_2.jpg,999_data/data/right_6133_4_R_2.jpg,1.0,0.0
159,6120,6120_7_LR_2,999_data/data/left_6120_7_L_2.jpg,999_data/data/right_6120_7_R_2.jpg,4.0,4.0
1348,6274,6274_4_LR,999_data/data/left_6274_4_L.jpg,999_data/data/right_6274_4_R.jpg,0.0,0.0


In [784]:
F_merged
LR_merged

,user_id,filename,left_path,right_path,left_label,right_label
0,6101,6101_4_LR,999_data/data/left_6101_4_L.jpg,999_data/data/right_6101_4_R.jpg,0.0,0.0
1,6101,6101_4_LR_2,999_data/data/left_6101_4_L_2.jpg,999_data/data/right_6101_4_R_2.jpg,0.0,0.0
2,6101,6101_5_LR,999_data/data/left_6101_5_L.jpg,999_data/data/right_6101_5_R.jpg,0.0,0.0
3,6101,6101_5_LR_2,999_data/data/left_6101_5_L_2.jpg,999_data/data/right_6101_5_R_2.jpg,0.0,0.0
4,6101,6101_6_LR,999_data/data/left_6101_6_L.jpg,999_data/data/right_6101_6_R.jpg,0.0,0.0
...,...,...,...,...,...,...
1575,6302,6302_5_LR_2,999_data/data/left_6302_5_L_2.jpg,999_data/data/right_6302_5_R_2.jpg,0.0,0.0
1576,6302,6302_6_LR,999_data/data/left_6302_6_L.jpg,999_data/data/right_6302_6_R.jpg,0.0,0.0
1577,6302,6302_6_LR_2,999_data/data/left_6302_6_L_2.jpg,999_data/data/right_6302_6_R_2.jpg,0.0,0.0
1578,6302,6302_7_LR,999_data/data/left_6302_7_L.jpg,999_data/data/right_6302_7_R.jpg,0.0,0.0


In [785]:
merged_df = pd.concat([F_merged, LR_merged], ignore_index=True)
merged_df

,user_id,filename,left_path,right_path,left_label,right_label
0,6101,6101_5_F,999_data/data/left_6101_5_F.jpg,999_data/data/right_6101_5_F.jpg,0.0,0.0
1,6101,6101_4_F_2,999_data/data/left_6101_4_F_2.jpg,999_data/data/right_6101_4_F_2.jpg,0.0,0.0
2,6101,6101_4_F,999_data/data/left_6101_4_F.jpg,999_data/data/right_6101_4_F.jpg,0.0,0.0
3,6101,6101_5_F_2,999_data/data/left_6101_5_F_2.jpg,999_data/data/right_6101_5_F_2.jpg,0.0,0.0
4,6101,6101_7_F,999_data/data/left_6101_7_F.jpg,999_data/data/right_6101_7_F.jpg,0.0,0.0
...,...,...,...,...,...,...
3155,6302,6302_5_LR_2,999_data/data/left_6302_5_L_2.jpg,999_data/data/right_6302_5_R_2.jpg,0.0,0.0
3156,6302,6302_6_LR,999_data/data/left_6302_6_L.jpg,999_data/data/right_6302_6_R.jpg,0.0,0.0
3157,6302,6302_6_LR_2,999_data/data/left_6302_6_L_2.jpg,999_data/data/right_6302_6_R_2.jpg,0.0,0.0
3158,6302,6302_7_LR,999_data/data/left_6302_7_L.jpg,999_data/data/right_6302_7_R.jpg,0.0,0.0


In [787]:
import json

json_path = os.path.join(pig_root, '999_data', 'label', 'pig_label.json')
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, 'w') as f:
    json.dump(merged_df.to_dict(orient='records'), f, indent=2, ensure_ascii=False)